# ASTD Dataset Audit

This notebook inspects the uploaded ASTD archive, verifies the official split files, documents class imbalance, and checks for exact-text duplicates across train/validation/test. The test split is preserved unchanged in the leakage-cleaning step.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from astd_loader import load_astd, load_official_unbalanced_splits, remove_exact_duplicate_leakage_preserve_test, LABEL_MEANING

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "ASTD-master" / "data"
print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)

In [ ]:
df = load_astd(DATA_DIR)
print("Shape:", df.shape)
display(df.head())
print("\nLabel counts:")
display(df["label"].value_counts().rename_axis("label").to_frame("count"))
print("\nLabel percentages:")
display((df["label"].value_counts(normalize=True) * 100).round(2).rename("percent").to_frame())

## Label interpretation

The uploaded file uses the codes `POS`, `NEG`, `NEUTRAL`, and `OBJ`. The original ASTD paper describes the categories as subjective positive, subjective negative, subjective mixed, and objective. Keep the raw code in the data; use the documented mapping in the report.

In [ ]:
label_table = (
    df["label"].value_counts()
      .rename_axis("raw_label")
      .to_frame("count")
      .assign(meaning=lambda x: x.index.map(LABEL_MEANING))
)
display(label_table)

## Verify official unbalanced partitions

In [ ]:
official = load_official_unbalanced_splits(DATA_DIR, df)
for name, part in official.items():
    print(f"{name:>10}: {len(part):>5} rows")
    display(part["label"].value_counts().rename_axis("label").to_frame(name))

sets = {name: set(part["row_id"]) for name, part in official.items()}
print("Train/validation overlap:", len(sets["train"] & sets["validation"]))
print("Train/test overlap:", len(sets["train"] & sets["test"]))
print("Validation/test overlap:", len(sets["validation"] & sets["test"]))
print("Union size:", len(set.union(*sets.values())))
print("Missing row IDs:", len(set(df["row_id"]) - set.union(*sets.values())))

## Duplicate-text and potential leakage audit

The official row-ID splits can be disjoint while identical tweet text appears in more than one split. We check exact duplicate text values and locate any cross-split collisions. This is important because a repeated text in training and test could inflate evaluation scores.

In [ ]:
duplicate_rows = df[df["text"].duplicated(keep=False)].sort_values("text")
print("Rows belonging to exact-duplicate text groups:", len(duplicate_rows))
print("Number of distinct exact-duplicate texts:", duplicate_rows["text"].nunique())
display(duplicate_rows)

for name, part in official.items():
    print(f"Duplicate texts within {name}:", int(part["text"].duplicated().sum()))

for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    shared = set(official[left]["text"]) & set(official[right]["text"])
    print(f"Exact duplicate texts across {left}/{right}:", len(shared))

## Leakage-controlled splits

For this project, the code below preserves every original test row unchanged. It removes exact duplicates from training/validation when the same text occurs in the held-out test set, and removes repeat copies within training/validation. Do not alter or augment the test set. Record this cleaning decision in the methodology.

In [ ]:
clean = remove_exact_duplicate_leakage_preserve_test(official)
for name, part in clean.items():
    print(f"{name:>10}: {len(part):>5} rows; duplicate texts: {part['text'].duplicated().sum()}")
    display(part["label"].value_counts().rename_axis("label").to_frame(name))

assert not (set(clean["train"]["text"]) & set(clean["validation"]["text"]))
assert not (set(clean["train"]["text"]) & set(clean["test"]["text"]))
assert not (set(clean["validation"]["text"]) & set(clean["test"]["text"]))
assert clean["test"].equals(official["test"].reset_index(drop=True))
print("Leakage checks passed; original test rows are unchanged.")

## Save audit-cleaned partitions locally

The following CSV files are generated under `data/processed/`. That directory is ignored by Git so the source dataset and derivative text data are not accidentally pushed to a public repository.

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for name, part in clean.items():
    out = PROCESSED_DIR / f"astd_{name}.csv"
    part.to_csv(out, index=False, encoding="utf-8-sig")
    print("Saved:", out)

## Initial decisions to take to the project team

1. Confirm ASTD is the final dataset and sentiment classification is the selected task.
2. Confirm the exact-duplicate removal policy while preserving the held-out test set.
3. Use Macro-F1 as the primary metric because the classes are highly imbalanced; report Accuracy, per-class Precision/Recall/F1, Weighted-F1, and the confusion matrix as secondary outputs.
4. Select low-resource training fractions (for example 10%, 25%, and 50%) from the cleaned training split using stratified sampling and fixed seeds.
5. Apply all augmentation methods to training samples only.